# Exploratory Data Analysis

This notebook is structured as follows:

1. **Load the data.**
2. **Class distribution.**
3. **Data quality.**
4. **Redundant features.**
5. **Duplicate rows.**
6. **Feature distributions.**
7. **Feature correlation.**
8. **Summary.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Load the data

The first step is to load the parquet file which we generated from `scripts/merge_data.py`. 
After we have loaded the data into a dataframe, we can display five rows and the statistics in order to get a quick insigth into the dataet.

In [ ]:
df = pd.read_parquet("../data/processed/cicids2017.parquet")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Class distribution

Now we explore the class distribution (on a log-scale) in order to get an insigth into how the data we have is distributed.

In [ ]:
class_counts = df[" Label"].value_counts()

ax = class_counts.plot(
    kind="bar",
    figsize=(12, 6),
    logy=True,
)

ax.set_title("CICIDS2017 Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of flows (log scale)")
ax.tick_params(axis="x", rotation=45)

ax.grid(axis="y", alpha=0.3, which="both")
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

Key insigths:

- The benign instances take the cake, with more than $10^6$ instances
- Less sophisiticated attacks like DoS, port scanning and DDoS make up the three most popular malicious classes
- Infiltration, SQL injection and Heartbleed are severely underrepresented with less than 100 instances each (Heartbleed with barely over 10)
- These rare attack samples can skew metrics and mask weak detection on minority classes


## Data quality

We count missing, infinite and negative values per feature.

In [ ]:
# Select numeric features only
numeric_df = df.select_dtypes(include="number")

# Count data-quality issues
quality = pd.DataFrame(
    {
        "Missing": numeric_df.isna().sum(),
        "Infinite": np.isinf(numeric_df).sum(),
        "Negative": (numeric_df < 0).sum(),
    }
)

# Add percentages
quality["Missing (%)"] = quality["Missing"] / len(df) * 100
quality["Infinite (%)"] = quality["Infinite"] / len(df) * 100
quality["Negative (%)"] = quality["Negative"] / len(df) * 100

# Sort by total number of problematic values
quality["Total issues"] = quality["Missing"] + quality["Infinite"] + quality["Negative"]

quality = quality.sort_values("Total issues", ascending=False)

quality.head(20)

Key insights:

- `Init_Win_bytes_forward` and `Init_Win_bytes_backward` are $-1$ in 35–50% of the rows. This is a sentinel meaning "not observed", not an error, so we keep it.
- The infinite and missing values in `Flow Bytes/s` and `Flow Packets/s` come from flows with zero duration (division by zero).
- Negative durations, IATs and header lengths are measurement artefacts. They affect about 0.1% of the rows, so we can safely drop them.

## Redundant features

Constant features carry no information, and features with identical contents are pure duplicates.

In [ ]:
constant_cols = [c for c in numeric_df.columns if numeric_df[c].min() == numeric_df[c].max()]
print("Constant columns:", constant_cols)

varying_cols = [c for c in numeric_df.columns if c not in constant_cols]
identical_pairs = [
    (a, b)
    for i, a in enumerate(varying_cols)
    for b in varying_cols[i + 1 :]
    if numeric_df[a].equals(numeric_df[b])
]
pd.DataFrame(identical_pairs, columns=["Column", "Identical to"])

Key insights:

- 8 columns are constant (all the `*Bulk*` features, `Bwd PSH Flags` and `Bwd URG Flags`) and can be dropped.
- Several columns are exact copies of others, e.g. `Fwd Header Length.1`, the `Subflow * Packets` columns, `SYN Flag Count` (= `Fwd PSH Flags`) and `CWE Flag Count` (= `Fwd URG Flags`). One of each pair can be dropped.

## Duplicate rows

We count identical rows, and rows whose features are identical to another row but with a different label.

In [ ]:
feature_cols = numeric_df.columns

n_duplicates = df.duplicated().sum()
n_conflicting = df.duplicated(subset=feature_cols).sum() - n_duplicates

print(f"Duplicate rows: {n_duplicates} ({n_duplicates / len(df):.1%})")
print(f"Same features, different label: {n_conflicting}")

Key insights:

- About 11% of the rows are exact duplicates. If they are not removed *before* the train/test split, identical samples end up in both sets and the test scores are inflated (data leakage).
- Rows with identical features but different labels cannot be classified correctly by any model and should be dropped.

## Feature distributions

To keep the plots fast, we work on a random sample of 200 000 rows from here on. We compare the raw distribution of a few representative features with their $\log(1 + x)$ transform.

In [ ]:
sample = numeric_df.sample(200_000, random_state=42)
sample.columns = sample.columns.str.strip()
sample = sample.replace([np.inf, -np.inf], np.nan)

features = ["Flow Duration", "Total Fwd Packets", "Flow Bytes/s", "Packet Length Mean"]

fig, axes = plt.subplots(2, len(features), figsize=(16, 7))
for j, feature in enumerate(features):
    values = sample[feature].dropna().clip(lower=0)
    axes[0, j].hist(values, bins=50)
    axes[0, j].set_title(feature)
    axes[1, j].hist(np.log1p(values), bins=50)
    axes[1, j].set_xlabel("log(1 + x)")
    for ax in axes[:, j]:
        ax.set_yscale("log")

axes[0, 0].set_ylabel("Count (raw)")
axes[1, 0].set_ylabel("Count (log-transformed)")
plt.tight_layout()
plt.show()

Key insights:

- The features are extremely heavy-tailed: most flows are short and small, but a few span many orders of magnitude.
- The $\log(1 + x)$ transform gives much more well-behaved distributions. Together with standardisation this matters for logistic regression and the FFNN, which are sensitive to feature scale. Random forests split on thresholds and are unaffected.

## Feature correlation

We compute the Pearson correlation between all non-constant features on the sample.

In [ ]:
corr = sample.drop(columns=[c.strip() for c in constant_cols]).corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=6)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson correlation")
ax.set_title("Feature correlation")
plt.tight_layout()
plt.show()

In [ ]:
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

high_corr = (
    upper.rename_axis(index="Feature 1", columns="Feature 2")
    .reset_index()
    .melt(
        id_vars="Feature 1",
        var_name="Feature 2",
        value_name="Correlation",
    )
    .dropna()
)

high_corr = high_corr[high_corr["Correlation"].abs() > 0.99].sort_values(
    "Correlation",
    key=abs,
    ascending=False,
)

print(f"Feature pairs with |correlation| > 0.99: {len(high_corr)}")
high_corr

Key insights:

- There are large blocks of strongly correlated features, e.g. the packet-length statistics, the IAT statistics and the forward/backward packet counts.
- Highly correlated features make logistic regression coefficients unstable and hard to interpret, and split the importance between features in a random forest. This should be kept in mind when discussing feature importance.

## Summary

The exploration motivates the following cleaning steps:

- Strip the leading whitespace from the column names, and fix the broken `�` character in the web attack labels.
- Replace infinite values with NaN and drop rows with missing values or negative durations/IATs (~0.1% of the data). Keep the $-1$ sentinel in `Init_Win_bytes_*`.
- Drop the constant columns and one column from each identical pair.
- Drop duplicate rows and rows with conflicting labels *before* splitting into train and test sets.
- Handle the severe class imbalance: use metrics such as precision, recall and PR-AUC instead of accuracy, use class weights, and consider grouping the rare attack classes (Infiltration, SQL Injection, Heartbleed) for multi-class classification.
- Apply the $\log(1 + x)$ transform and standardisation inside the model pipeline, fitted on the training set only.